# Directly-Trained Sensitivity Results

This notebook plots aggregate per-example accuracy against the number of assignments in the evaluation metadata. Rows are the model's training dataset and columns are the evaluation dataset, both ordered r-trivial, aperiodic, then periodic.

Each line is a model architecture and training-supervision level: transformer lines are red, hybrid lines are purple, and darker shades denote more supervision. Evaluation datasets and assignment counts are read from the downloaded request metadata rather than inferred from task names.

In [ ]:
from __future__ import annotations

from collections import defaultdict
import json
import re
import sys
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "aesthetics.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import aesthetics as aes  # noqa: E402

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 140)

## Config

In [ ]:
RESULTS_PATH = (
    PROJECT_ROOT
    / "results"
    / "jacksonp-directly-trained-sensitivity-evals"
    / "all_results.json"
)

# None pools the supervised and unsupervised evaluation variants for each
# evaluation dataset. Set to "supervised" or "unsupervised" to inspect one.
EVALUATION_SUPERVISION = None

# Draw each training distribution's parsed `*_ltNN` cutoff in its facet row.
SHOW_TRAINING_CUTOFFS = True

TRAIN_EVAL_ORDER = ["r-trivial", "aperiodic", "periodic"]
DATASET_LABEL = {
    "r-trivial": "R-trivial",
    "aperiodic": "Aperiodic",
    "periodic": "Periodic",
}
SUPERVISION_ORDER = [0, 50, 100]

# The hue encodes architecture first, then supervision by light-to-dark shade.
SERIES_PALETTE = {
    "Transformer · 0% supervision": "#f2a6b0",
    "Transformer · 50% supervision": "#df5c6d",
    "Transformer · 100% supervision": "#b21d35",
    "Hybrid · 0% supervision": "#d5b0e8",
    "Hybrid · 50% supervision": "#aa6bcb",
    "Hybrid · 100% supervision": "#763d9a",
}
SERIES_ORDER = list(SERIES_PALETTE)

SAVE_FIGURE = False
FIGURE_PATH = PROJECT_ROOT / "figures" / "dt_accuracy_by_assignments.png"

## Load Per-Example Accuracy and Evaluation Metadata

In [ ]:
TRAINING_METADATA_RE = re.compile(
    r"(?P<architecture>transformer|hybrid)-"
    r"(?P<training_dataset>r-trivial|aperiodic|periodic)_"
    r"(?P<supervision>0|50|100)supervision_",
    flags=re.IGNORECASE,
)
TRAINING_CUTOFF_RE = re.compile(
    r"(?:^|[_-])(?:assignments_)?lt(?P<cutoff>\d+)(?:[_-]|$)",
    flags=re.IGNORECASE,
)
EVAL_NAME_RE = re.compile(
    r"(?P<dataset>Aperiodic|Periodic|RTrivial)(?P<supervision>Supervised|Unsupervised)"
)
EVAL_DATASET_FROM_NAME = {
    "Aperiodic": "aperiodic",
    "Periodic": "periodic",
    "RTrivial": "r-trivial",
}


def training_metadata_for(result: dict) -> dict:
    haystack = f"{result.get('experiment_name', '')} {result.get('model', '')}"
    match = TRAINING_METADATA_RE.search(haystack)
    if match is None:
        raise ValueError(f"Could not parse training metadata from: {haystack}")
    cutoff_match = TRAINING_CUTOFF_RE.search(haystack)
    if cutoff_match is None:
        raise ValueError(f"Could not parse the training cutoff from: {haystack}")
    return {
        "architecture": match["architecture"].lower(),
        "training_dataset": match["training_dataset"].lower(),
        "training_supervision": int(match["supervision"]),
        "training_cutoff": int(cutoff_match["cutoff"]),
    }


def evaluation_metadata_for(doc: dict) -> dict:
    """Read the evaluation dataset and supervision variant from request metadata."""
    eval_name = doc.get("eval_name")
    match = EVAL_NAME_RE.fullmatch(str(eval_name))
    if match is None:
        raise ValueError(f"Unexpected eval_name in request metadata: {eval_name!r}")
    return {
        "evaluation_dataset": EVAL_DATASET_FROM_NAME[match["dataset"]],
        "evaluation_supervision": (
            "supervised" if match["supervision"] == "Supervised" else "unsupervised"
        ),
        "assignment_count": int(doc["assignment_count"]),
    }


def artifact_file_for_task(result: dict, task: dict, kind: str) -> Path | None:
    task_hash = task.get("task_hash")
    if not task_hash:
        return None
    root = RESULTS_PATH.parent / result["experiment_id"] / "main" / kind
    matches = sorted(root.glob(f"*/*_{task_hash[-6:]}-{kind}.jsonl"))
    return matches[0] if matches else None


def metric_value(instance_metrics: dict, metric: str | None) -> float | None:
    value = instance_metrics
    for key in (metric or "").split(":"):
        if not isinstance(value, dict) or key not in value:
            return None
        value = value[key]
    return float(value) if isinstance(value, int | float) else None

In [ ]:
raw_results = json.loads(RESULTS_PATH.read_text())
totals = defaultdict(lambda: {"score_sum": 0.0, "n": 0, "tasks": set(), "experiments": set()})
missing_artifacts = []

for result in raw_results:
    training = training_metadata_for(result)

    for task in result.get("tasks", []):
        prediction_path = artifact_file_for_task(result, task, "predictions")
        request_path = artifact_file_for_task(result, task, "requests")
        if prediction_path is None or request_path is None:
            missing_artifacts.append((result["experiment_id"], task["task"]))
            continue

        metric = task.get("primary_metric")
        with prediction_path.open() as prediction_file, request_path.open() as request_file:
            for prediction_line, request_line in zip(prediction_file, request_file, strict=True):
                prediction = json.loads(prediction_line)
                request = json.loads(request_line)
                if prediction.get("doc_id") != request.get("doc_id"):
                    raise ValueError(
                        f"doc_id mismatch for {result['experiment_id']} {task['task']}: "
                        f"prediction={prediction.get('doc_id')} request={request.get('doc_id')}"
                    )

                score = metric_value(prediction.get("instance_metrics", {}), metric)
                if score is None:
                    continue

                evaluation = evaluation_metadata_for(request.get("doc", {}))
                if (
                    EVALUATION_SUPERVISION is not None
                    and evaluation["evaluation_supervision"] != EVALUATION_SUPERVISION
                ):
                    continue

                key = (
                    training["architecture"],
                    training["training_dataset"],
                    training["training_supervision"],
                    training["training_cutoff"],
                    evaluation["evaluation_dataset"],
                    evaluation["assignment_count"],
                )
                total = totals[key]
                total["score_sum"] += score
                total["n"] += 1
                total["tasks"].add(task["task"])
                total["experiments"].add(result["experiment_id"])

assignment_rows = []
for key, total in totals.items():
    architecture, training_dataset, training_supervision, training_cutoff, evaluation_dataset, assignment_count = key
    assignment_rows.append(
        {
            "architecture": architecture.title(),
            "training_dataset": training_dataset,
            "training_supervision": training_supervision,
            "training_cutoff": training_cutoff,
            "evaluation_dataset": evaluation_dataset,
            "assignment_count": assignment_count,
            "accuracy": total["score_sum"] / total["n"],
            "examples": total["n"],
            "eval_tasks": len(total["tasks"]),
            "experiments": len(total["experiments"]),
        }
    )

assignment_df = pd.DataFrame(assignment_rows)
if assignment_df.empty:
    raise ValueError(f"No per-example results found under {RESULTS_PATH.parent}")

assignment_df["training_dataset"] = pd.Categorical(
    assignment_df["training_dataset"], categories=TRAIN_EVAL_ORDER, ordered=True
)
assignment_df["evaluation_dataset"] = pd.Categorical(
    assignment_df["evaluation_dataset"], categories=TRAIN_EVAL_ORDER, ordered=True
)
assignment_df["series"] = (
    assignment_df["architecture"]
    + " · "
    + assignment_df["training_supervision"].astype(str)
    + "% supervision"
)
assignment_df = assignment_df.sort_values(
    ["training_dataset", "training_cutoff", "evaluation_dataset", "architecture", "training_supervision", "assignment_count"]
).reset_index(drop=True)

print(f"Loaded {len(raw_results):,} experiments and aggregated {assignment_df['examples'].sum():,} examples.")
if missing_artifacts:
    print(f"Skipped {len(missing_artifacts):,} task(s) without both request and prediction artifacts.")

coverage_df = (
    assignment_df.groupby(
        ["architecture", "training_dataset", "training_supervision", "training_cutoff", "evaluation_dataset"],
        observed=True,
        as_index=False,
    )
    .agg(
        assignment_values=("assignment_count", "nunique"),
        min_assignments=("assignment_count", "min"),
        max_assignments=("assignment_count", "max"),
        examples=("examples", "sum"),
        eval_tasks=("eval_tasks", "max"),
    )
    .sort_values(["training_dataset", "training_cutoff", "evaluation_dataset", "architecture", "training_supervision"])
)
display(coverage_df)

## Accuracy by Assignment Count

Accuracy pools all examples at each assignment count. With the default configuration, supervised and unsupervised evaluation variants are pooled within each evaluation dataset.

In [ ]:
training_cutoffs = (
    assignment_df.groupby("training_dataset", observed=True)["training_cutoff"]
    .unique()
    .to_dict()
)

plot_df = assignment_df.copy()
plot_df["training_dataset"] = plot_df["training_dataset"].map(DATASET_LABEL)
plot_df["evaluation_dataset"] = plot_df["evaluation_dataset"].map(DATASET_LABEL)
dataset_label_order = [DATASET_LABEL[name] for name in TRAIN_EVAL_ORDER]
present_series = [series for series in SERIES_ORDER if series in set(plot_df["series"])]

fig = plt.figure(figsize=(9.6, 6))
grid = fig.add_gridspec(
    nrows=len(TRAIN_EVAL_ORDER),
    ncols=len(TRAIN_EVAL_ORDER),
    # left=0.08,
    # right=0.72,
    # bottom=0.09,
    # top=0.95,
    hspace=0.35,
    wspace=0.22,
)

axes = []
for row_index, training_label in enumerate(dataset_label_order):
    row_axes = []
    for col_index, evaluation_label in enumerate(dataset_label_order):
        share_axis = axes[0][0] if axes else None
        ax = fig.add_subplot(grid[row_index, col_index], sharex=share_axis, sharey=share_axis)
        row_axes.append(ax)

        panel_df = plot_df[
            (plot_df["training_dataset"] == training_label)
            & (plot_df["evaluation_dataset"] == evaluation_label)
        ]
        if not panel_df.empty:
            sns.lineplot(
                data=panel_df,
                x="assignment_count",
                y="accuracy",
                hue="series",
                hue_order=present_series,
                palette=SERIES_PALETTE,
                marker="o",
                dashes=False,
                estimator=None,
                errorbar=None,
                legend=False,
                ax=ax,
            )

        if SHOW_TRAINING_CUTOFFS:
            training_dataset = TRAIN_EVAL_ORDER[row_index]
            for cutoff in sorted(training_cutoffs.get(training_dataset, [])):
                ax.axvline(cutoff, color="0.35", linestyle=":", linewidth=1.0, zorder=1)

        ax.set_title(f"Train: {training_label} | Eval: {evaluation_label}", fontsize=8, fontweight="regular")
        ax.set_ylim(-0.02, 1.02)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=5, integer=True))
        ax.set_xlabel("# assignments" if row_index == len(TRAIN_EVAL_ORDER) - 1 else "")
        ax.set_ylabel("Aggregate accuracy" if col_index == 0 else "")
        ax.tick_params(labelbottom=row_index == len(TRAIN_EVAL_ORDER) - 1, labelleft=col_index == 0)
    axes.append(row_axes)

legend_handles = [
    plt.Line2D([0], [0], color=SERIES_PALETTE[series], marker="o", linewidth=1.5, label=series)
    for series in present_series
]
fig.legend(
    handles=legend_handles,
    title="Architecture / training supervision",
    loc="center left",
    bbox_to_anchor=(0.74, 0.5),
    frameon=False,
)

if SAVE_FIGURE:
    FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_PATH, dpi=300, bbox_inches="tight")
    print(f"Saved figure to {FIGURE_PATH}")

plt.show()